# Modules => Packages & the `__main__` Guard

A **module** is one `.py` file. A **package** is a folder of modules. This notebook covers packages, how imports are cached, and how a file can be both importable and runnable.

| Concept | Meaning |
|---|---|
| Module | A single `.py` file |
| Package | A folder that contains modules |
| `__init__.py` | Marks and initializes a regular package. Runs on first import |
| `__name__` | The module's name. It is `"__main__"` when the file is run directly |
| `if __name__ == "__main__":` | Runs code only when the file is executed, not imported |
| `__all__` | Names exported by `from module import *` |
| `sys.modules` | Cache of already imported modules |
| `importlib.reload()` | Re-imports a module after it changed |
| `from . import x` | Relative import inside a package |
| `python -m package.module` | Runs a module as a script |

---

## Package Structure

```text
shop/
    __init__.py
    prices.py
    reports/
        __init__.py
        summary.py
```

```python
import shop.prices
from shop import prices
from shop.reports.summary import make_summary
```

### `__init__.py`

* Runs **once**, the first time the package is imported.
* Can be empty.
* Can re-export names so users write `from shop import total` instead of `from shop.prices import total`.

---

## Relative Imports

Inside a package, `.` means "this package" and `..` means "the parent package".

| Import | Meaning |
|---|---|
| `from . import prices` | A sibling module |
| `from .prices import total` | A name from a sibling module |
| `from ..prices import total` | A name from the parent package |

Relative imports only work **inside packages**. They fail in a script that is run directly.

---

## The `__main__` Guard

Every module has a `__name__`.

| How the file is used | Value of `__name__` |
|---|---|
| Imported | The module name, such as `"shop.prices"` |
| Run directly (`python file.py`) | `"__main__"` |

```python
def main():
    print("running")

if __name__ == "__main__":
    main()
```

The guard lets one file work as a **library** (import it) and as a **program** (run it).

---

## `__all__`

Controls what `from module import *` exports:

```python
__all__ = ["total", "average"]
```

Names that are not listed are not imported by `*`. Direct imports such as `from module import hidden` still work.

---

## Import Caching

* Python runs a module **once** and stores it in `sys.modules`.
* A second `import` reuses the stored module. It does **not** run the file again.
* After editing a module, use `importlib.reload(module)` or restart the interpreter.

---

## Running Modules and Packages

| Command | Meaning |
|---|---|
| `python file.py` | Run a file. `__name__` is `"__main__"` |
| `python -m package.module` | Run a module by its import path. Relative imports work |
| `python -m package` | Run the package's `__main__.py` |

---

## Common Problems

| Problem | Cause |
|---|---|
| Your file shadows a standard module | A file named `random.py` or `json.py` hides the real one. Rename it |
| `ImportError` in a relative import | The file was run directly instead of with `-m` |
| Changes are not visible | The module is cached. Reload or restart |
| Circular import | Two modules import each other. Move shared code to a third module |

## Source

https://docs.python.org/3/tutorial/modules.html

https://docs.python.org/3/reference/import.html

In [ ]:
import importlib
import runpy
import sys
import tempfile
from pathlib import Path

# Build a small package on disk: shop/ with a sub-package
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "shop" / "reports").mkdir(parents=True)

    (root / "shop" / "__init__.py").write_text(
        'print("shop/__init__.py runs once")\n'
        "from .prices import total\n"
        '__all__ = ["total"]\n'
    )
    (root / "shop" / "prices.py").write_text(
        'print("prices.py runs once, __name__ =", __name__)\n'
        "def total(items):\n"
        "    return sum(items)\n"
    )
    (root / "shop" / "cli.py").write_text(
        "from .prices import total\n"
        "\n"
        "def main():\n"
        '    print("cli main:", total([1, 2, 3]))\n'
        "\n"
        'print("cli.py loaded, __name__ =", __name__)\n'
        'if __name__ == "__main__":\n'
        "    main()\n"
    )
    (root / "shop" / "reports" / "__init__.py").write_text("")
    (root / "shop" / "reports" / "summary.py").write_text(
        "from ..prices import total\n"          # relative import from the parent package
        "def summary(items):\n"
        '    return f"total={total(items)}"\n'
    )

    sys.path.insert(0, tmp)
    try:
        import shop                                # runs __init__.py and prices.py once
        from shop.reports.summary import summary   # relative import works inside the package
        print(shop.total([10, 20]), summary([1, 2, 3]))
        print(shop.__all__)

        # Import caching: importing again does not run the files again
        import shop as again
        print(again is shop, "shop.prices" in sys.modules)

        # Reload runs the module code again
        importlib.reload(shop.prices)

        # Run a module as a script: __name__ becomes "__main__", so main() runs
        runpy.run_module("shop.cli", run_name="__main__")

        # Import the same module: __name__ is the module name, so main() does NOT run
        import shop.cli
    finally:
        sys.path.remove(tmp)
        for name in [m for m in sys.modules if m == "shop" or m.startswith("shop.")]:
            del sys.modules[name]

print("shop" in sys.modules)                       # cleaned up
print(__name__)                                    # the notebook or script itself is "__main__"